# 07 — DistilBERT: SHAP-Guided Counterfactual Selective Correction

**Research project:** Explainability-Guided Feature Analysis for Reducing False Positives in Phishing Email Detection Without Increasing False Negatives

This notebook examines a SHAP-guided selective-correction procedure applied to the official DistilBERT baseline. The aim was to reduce legitimate emails incorrectly classified as phishing while keeping the number of phishing emails incorrectly classified as legitimate from increasing. The procedure adjusted selected predictions without retraining the model.

## 1. Research objective

The official DistilBERT baseline produced **31 false positives** and **22 false negatives** on the fixed test set. The experiment investigated whether counterfactual changes to model-visible text could identify false-positive predictions suitable for selective correction.

The success criterion was **fewer than 31 false positives**, with **no more than 22 false negatives**. DistilBERT's trained weights and its decision threshold remained unchanged.

## 2. Dataset and classification terms

**Table 7.1. Dataset used in the experiment**

| Dataset component | Number of emails |
|:---|---:|
| Cleaned dataset | 53,577 |
| Training set | 42,861 |
| Test set | 10,716 |

The classification labels were **0 = legitimate** and **1 = phishing**. The decision threshold was **0.50**.

The project uses the following classification terminology: **TP** refers to a legitimate email correctly classified as legitimate; **FP** refers to a legitimate email incorrectly classified as phishing; **FN** refers to a phishing email incorrectly classified as legitimate; and **TN** refers to a phishing email correctly classified as phishing. These definitions are retained in all classification tables.

## 3. SHAP-guided selective-correction method

SHAP analysis was used to investigate text patterns associated with the baseline model's false-positive predictions. Candidate patterns were assessed before being used in the selective-correction procedure.

For an email predicted as phishing, the procedure searched its model-visible text for retained patterns. When a pattern was present, a separate counterfactual version of the text was created by removing that pattern. The original DistilBERT model then classified the modified text. A change from a phishing prediction to a legitimate prediction provided the condition for selective correction. Where several retained patterns matched, the patterns were tested separately.

The original email was preserved. The method changed selected final decisions; it **did not update the model weights**, retrain DistilBERT, or alter the 0.50 decision threshold.

**Table 7.2. Controls applied during selective correction**

| Experimental component | Treatment |
|:---|:---|
| DistilBERT model weights | Kept unchanged |
| Classification threshold | Kept at 0.50 |
| Original email text | Preserved |
| Counterfactual text changes | Applied to temporary evaluation copies only |
| Candidate patterns | Screened before use |
| Selective decision correction | Applied when a retained pattern met the counterfactual condition |

## 4. Classification results

**Table 7.3. Classification outcomes on the fixed test set**

| Classification outcome | Official baseline | Selective correction | Change |
|:---|---:|---:|---:|
| TP — Correctly classified legitimate | 4,717 | 4,725 | +8 |
| FP — Legitimate classified as phishing | **31** | **23** | **−8** |
| FN — Phishing classified as legitimate | **22** | **22** | **0** |
| TN — Correctly classified phishing | 5,946 | 5,946 | 0 |
| **Total evaluated emails** | **10,716** | **10,716** | **0** |

Eight legitimate emails that had been incorrectly classified as phishing were corrected. The number of false negatives remained unchanged. The procedure therefore met the experiment's error-count criterion on the evaluated test set.

**Table 7.4. Confusion matrices (actual class by predicted class)**

| Model | Actual class | Predicted legitimate (0) | Predicted phishing (1) | Total |
|:---|:---|---:|---:|---:|
| Official baseline | Legitimate (0) | 4,717 | 31 | 4,748 |
| Official baseline | Phishing (1) | 22 | 5,946 | 5,968 |
| Selective correction | Legitimate (0) | 4,725 | 23 | 4,748 |
| Selective correction | Phishing (1) | 22 | 5,946 | 5,968 |

The row totals confirm that both methods were evaluated on the same **4,748 legitimate** and **5,968 phishing** emails.

In [ ]:
# Verify the recorded confusion-matrix counts and calculate standard metrics.
# Standard precision and recall below use phishing (label 1) as the positive class.

baseline = {"TP": 4717, "FP": 31, "FN": 22, "TN": 5946}
selective = {"TP": 4725, "FP": 23, "FN": 22, "TN": 5946}

def evaluate(counts):
    tp, fp, fn, tn = (counts[k] for k in ("TP", "FP", "FN", "TN"))
    total = tp + fp + fn + tn
    precision = tn / (tn + fp)  # phishing is label 1
    recall = tn / (tn + fn)
    return {
        "Total": total,
        "Accuracy": (tp + tn) / total,
        "Precision": precision,
        "Recall": recall,
        "F1-score": 2 * precision * recall / (precision + recall),
    }

assert evaluate(baseline)["Total"] == evaluate(selective)["Total"] == 10716
assert selective["FP"] < baseline["FP"]
assert selective["FN"] <= baseline["FN"]
for name, counts in (("Official baseline", baseline), ("Selective correction", selective)):
    result = evaluate(counts)
    print(f"{name}: " + ", ".join(f"{k}={v:.6f}" if isinstance(v, float) else f"{k}={v:,}" for k, v in result.items()))

## 5. Performance comparison

**Table 7.5. Classification performance**

| Performance measure | Official baseline | Selective correction |
|:---|---:|---:|
| Accuracy | 0.995054 | 0.995801 |
| Precision (phishing class) | 0.994813 | 0.996147 |
| Recall (phishing class) | 0.996314 | 0.996314 |
| F1-score (phishing class) | 0.995563 | 0.996230 |

Precision, recall, and F1-score use **phishing (class 1)** as the positive class. The notebook calculates these measures directly from the recorded classification counts.

**Table 7.6. Changes in classification errors**

| Error measure | Official baseline | Selective correction | Change |
|:---|---:|---:|:---|
| False positives | 31 | **23** | 8 fewer (25.81%) |
| False negatives | 22 | **22** | No change |
| **Total errors** | **53** | **45** | **8 fewer (15.09%)** |

The procedure improved precision and accuracy by correcting eight false positives, while phishing recall remained unchanged.

## 6. Interpretation of the findings

The results show that SHAP-guided counterfactual testing identified eight false-positive predictions that could be corrected without increasing false negatives on the fixed test set. The approach therefore satisfied the stated objective for this experiment.

This was a **prediction-time selective-correction method**, rather than an additional training stage. The retained patterns were selected and assessed using the experiment's error-analysis results. Accordingly, the reported improvement describes performance on the evaluated emails and should be distinguished from an evaluation on a separate, previously unused dataset.

## 7. Conclusion

SHAP-guided counterfactual selective correction reduced false positives from **31 to 23**, while false negatives remained at **22**. This represents a **25.81% reduction in false positives** without retraining DistilBERT or changing its classification threshold.

The experiment demonstrated that targeted examination of model predictions can improve the observed classification results. It also motivated subsequent experiments using corrective training examples to refine the model itself.